# Rule: **build_daily_heat_demand**

**Description**

This rule builds a proxy for the heat demand time series using heating degree day (HDD) approximation: the daily gap between the indoor comfort temperature and the outdoor temperature, counted only when the outside is colder. A cold day yields a large HDD value; a mild day yields zero.

For this purpose, the mean daily temperature is used (not the hourly temperature). The comfort temperature is set to the default value of 15°C.

Snapshots are resampled to daily time resolution and ``Atlite.convert.heat_demand`` is used to convert ambient temperature from the weather cutout selected in `sector: heat_demand_cutout` to heat demand time series.

Heat demand is distributed by population to clustered onshore regions.

The result is, for each day and each region, a single number representing the total population-weighted heating need driven by that day's weather. 

**NOTE: The output is a profile with no energy units**

**Outputs**

- resources/`daily_heat_demand_total_base_s_{clusters}.nc`

In [ ]:
######################################## Parameters

### Run
name = 'case_SectorCoupled'
prefix = ''

### Network
clusters = 5

In [ ]:
##### Import packages
import xarray as xr
import os
import sys
import matplotlib.pyplot as plt
import pandas as pd

##### Import local functions
sys.path.append(os.path.abspath(os.path.join('..')))
import functions as xp


##### Read params.yaml
params = xp.read_params('../params.yaml')


##### Ignore warnings
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

## `daily_heat_demand_total_base_s_{clusters}.nc`

Load the dataset and show its structure.

In [ ]:
file = f'daily_heat_demand_total_base_s_{clusters}.nc'
path = f'{params["rootpath"]}/resources/{prefix}/{name}/'

ds = xr.open_dataset(path + file)

ds

Plot total daily heat demand aggregated over nodes for all variables in the dataset.

In [ ]:
time_dim = 'time'
node_dim = 'name'

fig, ax = plt.subplots(figsize=(12, 5))

for v in ds.data_vars:
    da = ds[v]
    if node_dim is not None and node_dim in da.dims:
        series = da.sum(dim=node_dim).to_series()
    else:
        series = da.to_series()
    series = series.sort_index()
    series.plot(ax=ax, label=v, linewidth=1.0, alpha=0.85)

ax.set_title('Daily heat demand profile, aggregated over nodes')
ax.set_xlabel('Date')
ax.set_ylabel('[-]')
ax.grid(True, linestyle='--', alpha=0.4)
ax.legend(title='Variable', ncol=2)
plt.tight_layout()

Plot total daily heat demand broken down according to nodes.

In [ ]:
# Stacked area chart by node (same variable as previous plot)
selected_var = 'heat_demand'
da = ds[selected_var]

fig, ax = plt.subplots(figsize=(14, 6))

if 'time' in da.dims and 'name' in da.dims:
    df = da.transpose('time', 'name').to_pandas().sort_index()
    ax.stackplot(df.index, [df[col].values for col in df.columns], labels=[str(c) for c in df.columns], alpha=0.85)
    ax.legend(title='Node', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
else:
    series = da.to_series().sort_index()
    ax.fill_between(series.index, series.values, alpha=0.5, label=selected_var)
    ax.legend()

ax.set_title('Daily heat demand profile by node (stacked area)')
ax.set_xlabel('Date')
ax.set_ylabel('[-]')
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()

Plot daily heat demand time series for each node.

In [ ]:
# Superposed time series without node aggregation (one line per node)
selected_var = 'heat_demand'
da = ds[selected_var]

fig, ax = plt.subplots(figsize=(14, 6))

if 'time' in da.dims and 'name' in da.dims:
    df = da.transpose('time', 'name').to_pandas().sort_index()
    for col in df.columns:
        ax.plot(df.index, df[col], linewidth=0.9, alpha=0.8, label=str(col))
    ax.legend(title='Node', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
else:
    series = da.to_series().sort_index()
    ax.plot(series.index, series.values, linewidth=1.2, label=selected_var)
    ax.legend()

ax.set_title(f'Daily heat demand profile by node')
ax.set_xlabel('Date')
ax.set_ylabel('[-]')
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()